# Conditional GAN — generation you can steer with a label

> Tutorial pair for [`conditional_gan.py`](conditional_gan.py).

## 1. Intuition
A plain GAN samples from $p(x)$ with no say over *which* sample you get. A
**conditional GAN** hands both the generator and the discriminator an extra label
$y$, so the model learns $p(x\mid y)$ and you can ask it for "a sample of class 3".
The discriminator's job gets sharper too: not "is this real?" but "is this a real
example *of class $y$*?".

## 2. Concept (the slide)
- A label $y$ is mapped to a learned **embedding** vector and concatenated into
  the inputs of both nets.
- **Generator** $G(z, y)$: noise + label embedding $\to$ a class-conditional sample.
- **Discriminator** $D(x, y)$: sample + label embedding $\to$ logit "real *and* of
  class $y$".
- The adversarial loss is the **non-saturating GAN loss**, but every term is now
  conditioned on $y$.

## 3. Math derivation — conditioning the minimax game

Everything is conditioned on the label $y$. The value function becomes
$$\min_G\max_D\;V(D,G)=
 \mathbb E_{(x,y)\sim p_{\text{data}}}[\log D(x\mid y)]
 +\mathbb E_{y\sim p(y),\,z\sim p_z}[\log(1-D(G(z\mid y)\mid y))].$$

Fixing $G$ and maximizing pointwise *per label* gives the conditional optimal
discriminator
$$D^\star(x\mid y)=\frac{p_{\text{data}}(x\mid y)}{p_{\text{data}}(x\mid y)+p_g(x\mid y)},$$
and substituting it back shows the generator minimizes the **expected
Jensen–Shannon divergence over labels**
$$\mathbb E_{y\sim p(y)}\big[\mathrm{JSD}\big(p_{\text{data}}(\cdot\mid y)\,\Vert\,p_g(\cdot\mid y)\big)\big]
 \;-\;\log 4,$$
which is zero iff $p_g(x\mid y)=p_{\text{data}}(x\mid y)$ for every $y$ — i.e. the
generator matches the data distribution **class by class**.

**Why this differs from vanilla GAN.** The vanilla GAN matches only the marginal
$p(x)$, so a sample's class is uncontrollable and modes can be dropped. The
conditional version matches each conditional $p(x\mid y)$, which (a) gives explicit
control and (b) supplies the discriminator with the label as side information,
making "real but wrong class" detectable and discouraging class-level mode collapse.

In code, both nets use `nn.Embedding(n_classes, emb_dim)` and the trainer feeds
the *true* $y$ with real samples and a *sampled* $y$ with fakes.

## 4. Generator / key component

In [ ]:
# ===== actual implementation from conditional_gan.py =====
from __future__ import annotations

import numpy as np

import torch

import torch.nn as nn

SEED = 0

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def make_clusters(n: int = 2000, k: int = 4, r: float = 2.0, seed: int = SEED):
    rng = np.random.default_rng(seed)
    labels = rng.integers(0, k, n)
    ang = 2 * np.pi * labels / k
    centers = np.c_[r * np.cos(ang), r * np.sin(ang)]
    x = (centers + 0.1 * rng.normal(size=(n, 2))).astype(np.float32)
    return x, labels.astype(np.int64)

class Discriminator(nn.Module):
    def __init__(self, n_classes: int = 4, emb_dim: int = 8, data_dim: int = 2,
                 hidden: int = 64):
        super().__init__()
        self.emb = nn.Embedding(n_classes, emb_dim)
        self.net = nn.Sequential(
            nn.Linear(data_dim + emb_dim, hidden), nn.LeakyReLU(0.2, True),
            nn.Linear(hidden, hidden), nn.LeakyReLU(0.2, True),
            nn.Linear(hidden, 1))

    def forward(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        return self.net(torch.cat([x, self.emb(y)], dim=1))

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    k = 4
    x, y = make_clusters(2000, k=k)
    centers = np.array([[2 * np.cos(2 * np.pi * c / k), 2 * np.sin(2 * np.pi * c / k)]
                        for c in range(k)], dtype=np.float32)

    gan = ConditionalGANTorch(n_classes=k).fit(x, y, steps=1500, batch=128)
    print(f"D loss {np.mean(gan.d_hist[:100]):.3f} -> {np.mean(gan.d_hist[-100:]):.3f}")
    # Conditioning check: each requested class should land near its own center.
    ok = 0
    for c in range(k):
        s = gan.generate(200, label=c)
        d = np.linalg.norm(s.mean(0) - centers[c])
        near = int(np.argmin(np.linalg.norm(s.mean(0) - centers, axis=1)) == c)
        ok += near
        print(f"  class {c}: sample mean={s.mean(0).round(2)} "
              f"target={centers[c].round(2)} dist={d:.2f} "
              f"{'OK' if near else 'MISS'}")
    print(f"conditioning correct for {ok}/{k} classes")


class Generator(nn.Module):
    def __init__(self, noise_dim: int = 8, n_classes: int = 4, emb_dim: int = 8,
                 data_dim: int = 2, hidden: int = 64):
        super().__init__()
        self.emb = nn.Embedding(n_classes, emb_dim)
        self.net = nn.Sequential(
            nn.Linear(noise_dim + emb_dim, hidden), nn.LeakyReLU(0.2, True),
            nn.Linear(hidden, hidden), nn.LeakyReLU(0.2, True),
            nn.Linear(hidden, data_dim))

    def forward(self, z: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        return self.net(torch.cat([z, self.emb(y)], dim=1))

## 5. Trainer / losses

In [ ]:
# ===== actual implementation from conditional_gan.py =====
class ConditionalGANTorch:
    def __init__(self, noise_dim: int = 8, n_classes: int = 4, data_dim: int = 2,
                 lr: float = 2e-4):
        torch.manual_seed(SEED)
        self.dev = get_device()
        self.noise_dim, self.n_classes = noise_dim, n_classes
        self.G = Generator(noise_dim, n_classes, data_dim=data_dim).to(self.dev)
        self.D = Discriminator(n_classes, data_dim=data_dim).to(self.dev)
        self.optG = torch.optim.Adam(self.G.parameters(), lr=lr, betas=(0.5, 0.999))
        self.optD = torch.optim.Adam(self.D.parameters(), lr=lr, betas=(0.5, 0.999))
        self.bce = nn.BCEWithLogitsLoss()

    def fit(self, real: np.ndarray, labels: np.ndarray, steps: int = 1500, batch: int = 128):
        real = torch.as_tensor(real, dtype=torch.float32, device=self.dev)
        labels = torch.as_tensor(labels, dtype=torch.long, device=self.dev)
        ones = torch.ones(batch, 1, device=self.dev)
        zeros = torch.zeros(batch, 1, device=self.dev)
        self.d_hist, self.g_hist = [], []
        for _ in range(steps):
            idx = torch.randint(0, len(real), (batch,), device=self.dev)
            x, y = real[idx], labels[idx]
            # --- D step: (real, true y) -> 1, (fake, sampled y) -> 0 ---
            z = torch.randn(batch, self.noise_dim, device=self.dev)
            yf = torch.randint(0, self.n_classes, (batch,), device=self.dev)
            fake = self.G(z, yf).detach()
            lossD = self.bce(self.D(x, y), ones) + self.bce(self.D(fake, yf), zeros)
            self.optD.zero_grad(); lossD.backward(); self.optD.step()
            # --- G step: non-saturating, make (fake, yf) look real ---
            z = torch.randn(batch, self.noise_dim, device=self.dev)
            yf = torch.randint(0, self.n_classes, (batch,), device=self.dev)
            lossG = self.bce(self.D(self.G(z, yf), yf), ones)
            self.optG.zero_grad(); lossG.backward(); self.optG.step()
            self.d_hist.append(lossD.item()); self.g_hist.append(lossG.item())
        return self

    @torch.no_grad()
    def generate(self, n: int, label: int) -> np.ndarray:
        z = torch.randn(n, self.noise_dim, device=self.dev)
        y = torch.full((n,), int(label), dtype=torch.long, device=self.dev)
        return self.G(z, y).cpu().numpy()

## 6. Train

In [ ]:
demo()

## 7. Visualization

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
import conditional_gan as M

k = 4
x, y = M.make_clusters(2000, k=k)
gan = M.ConditionalGANTorch(n_classes=k).fit(x, y, steps=1500, batch=128)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(x[:, 0], x[:, 1], s=6, alpha=.15, color="gray", label="real")
colors = ["C0", "C1", "C2", "C3"]
for c in range(k):
    s = gan.generate(200, label=c)
    ax.scatter(s[:, 0], s[:, 1], s=8, alpha=.6, color=colors[c], label=f"gen class {c}")
ax.set_title("Each requested label lands on its own cluster"); ax.legend()
ax.set_aspect("equal")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- Conditioning turns $p(x)$ into $p(x\mid y)$: the generator becomes steerable and
  the discriminator matches each class distribution separately.
- Label embeddings concatenated to inputs are the simplest mechanism; richer
  schemes (projection discriminator, conditional BatchNorm) scale better.
- Pitfalls: if labels for fakes aren't sampled from the true $p(y)$, the model
  learns a skewed conditional; a too-strong D can still collapse *within* a class.